In [1]:
import requests
from bs4 import BeautifulSoup
import sqlite3

url="https://internshala.com/internships/"
response=requests.get(url)
print(response.status_code)
if response.status_code != 200:
    print("Failed to Login Page Error" ,response.status_code)

200


In [2]:
soup = BeautifulSoup(response.text, "html.parser")
jobs = soup.find_all("div", class_="individual_internship")
print(f" In This website I found {len(jobs)} job listing")

 In This website I found 51 job listing


In [3]:
con=sqlite3.connect("internship.db")
cursor=con.cursor()

cursor.execute("""
    CREATE  TABLE IF NOT EXISTS internships(
        id    INTEGER PRIMARY KEY AUTOINCREMENT,
        title    TEXT,
        company  TEXT,
        location TEXT,
        stipend  TEXT,
        duration  TEXT
        )
    """)

cursor.execute("DELETE FROM internships")
print("Table Created")


Table Created


In [4]:
def get_text(tag):
    if tag:
        return tag.get_text(separator=" ").strip()
    return "Not Satisfied"

def stipend_S(stipend):
    text=stipend
    number=""
    for ch in text:
        if ch.isdigit():
            number+=ch
    if number:
       return int(number)
    return "0"

def pass_Filter(title,company,location,stipend,duration):
    if FILTER_LOCATION and FILTER_LOCATION.lower()  not in location.lower():
        return False
    if FILTER_STIPEND >0:
        stipend_value=stipend_S(stipend)
        if stipend_value < FILTER_STIPEND:
            return False

    if FILTER_DURATION and FILTER_DURATION.lower() not in duration.lower():
        return False
    return True



In [5]:
result = []
rejected=0
saved=0
FILTER_LOCATION = input("Enter location: ")
FILTER_STIPEND = int(input("Enter min stipend: "))
FILTER_DURATION=input("Enter the duration :")



for job in jobs[:51]:

    title_tag=job.find("a",class_="job-title-href")
    company_tag = job.find("p",class_="company-name")
    location_tag = job.find("div", class_="row-1-item locations")
    stipend_tag=job.find("span",class_="stipend")
    duration_tag=job.find("i",class_="ic-16-calendar")
    

    title=get_text(title_tag)
    company=get_text(company_tag)
    location=get_text(location_tag)
    stipend=get_text(stipend_tag)
    duration=get_text(duration_tag.find_next("span")) if duration_tag else "Not satisfied"

   
    if not pass_Filter(title,company,location,stipend,duration):
        rejected+=1
        continue

    cursor.execute("""
        INSERT INTO internships (title, company, location, stipend, duration)
        VALUES (?, ?, ?, ?, ?)
    """, (title, company, location, stipend, duration))
 
    saved += 1
    # print(f"Saved: {title} — {company}")
    result.append({
            "title":    title,
            "company":  company,
            "location": location,
            "stipend":  stipend,
            "duration": duration,
        })
cursor.execute("SELECT * FROM internships")
rows = cursor.fetchall()

for row in rows:
    print(row)


con.commit()
con.close()

print(f"After filltering :{len(result)} matched , {rejected} skipped\n")

if not result:
        print("No internship match with your fillter")
else:
        for i,r in enumerate(result,start=1):
            print(f"[{i}] {r["title"]}")
            print(f"    Company:  {r["company"]}")
            print(f"    Location: {r["location"]}")
            print(f"    stipend:  {r["stipend"]}")
            print(f"    duration: {r["duration"]}")
            print("-" * 45)

Enter location:  
Enter min stipend:  0
Enter the duration : 


(1, 'Graphic Design', 'Stashpro', 'Pune', '₹ 12,000 - 20,000 /month', '6 Months')
(2, 'Inside Sales', 'YES Germany', 'Balewadi', '₹ 7,000 - 12,000 /month', '6 Months')
(3, 'Digital Marketing', 'ONergy Solar', 'Kolkata', '₹ 5,000 - 8,000 /month', '6 Months')
(4, 'Sales and Marketing', 'Anirah Advisory Llp', 'Ahmedabad', '₹ 9,000 - 35,000 /month', '6 Months')
(5, 'Video Editing/Making', 'TechDr Healthcare', 'Hyderabad', '₹ 10,000 - 12,000 /month', '6 Months')
(6, 'Digital Marketing & Inside Sales', 'Gearo Business Solutions', 'Pune, Pimpari', '₹ 8,000 - 12,000 /month', '6 Months')
(7, 'Recruitment', 'ONergy Solar', 'Kolkata', '₹ 3,000 - 5,000 /month', '4 Months')
(8, 'Business Devlopment Executive', 'Future Ready Learnings', 'Patna', '₹ 10,000 - 25,000 /month', '3 Months')
(9, 'Talent Acquisition', 'VST Tillers Tractors Limited', 'Bangalore', '₹ 9,000 - 10,000 /month', '6 Months')
(10, 'Graphic Design', 'StoryMirror Infotech Private Limited', 'Mumbai', '₹ 8,000 - 12,000 /month', '3 Month

In [6]:
import csv
with open("Internship.csv","w",newline="",encoding="utf-8") as f:
    writer=csv.DictWriter(f,fieldnames=["title","company","location","duration","stipend"])
    writer.writeheader()
    writer.writerows(result)
print(f"\n Done ! saved {len(result)} internship in internship.csv")


 Done ! saved 51 internship in internship.csv
